Building agents that process sensor data and maintain internal state. Design a simplified robotic vacuum and test its behavior in a discrete grid space.

## PEAS Formulation: 
Before writing your code, define the PEAS (Performance, Environment, Actuators, Sensors) specification for your vacuum. Use a performance metric of +10 points for every piece of dirt cleaned, and -1 point for every movement step. Your actuators are limited to Up, Down, Left, Right, and Suck.

## Environment Setup:
Program a 5x5 2D array to represent the room. Populate this environment by randomly assigning 5 tiles as "dirty" (represented by the value 1), while leaving the rest "clean" (represented by 0). 

## Place your agent at a origin.

## Agent A -
### The Simple Reflex Agent:
Implement a purely reactive agent that operates only on its current percept. If the agent's current tile is dirty, it must execute the "Suck" action (changing the tile to 0). If the tile is clean, it must pick a random, valid adjacent tile and move there (similar to your previous random walk).

## Agent B -
### The Model-Based Reflex Agent: 
Implement a stateful agent that maintains an internal memory of its environment. This agent follows the same basic cleaning rules as Agent A, but it keeps a running list of the (x, y) coordinates it has already visited. When choosing a direction to move from a clean tile, it must prioritize adjacent tiles that are not in its memory. It should only default to a random move if all adjacent tiles have already been visited.


# PEAS Formulation for the Vacuum Agent

## Performance Measure

+10 points for every dirty tile cleaned.
−1 point for every movement action (Up, Down, Left, Right).

## Goal: maximize cleaning score while minimizing unnecessary movement.

## Environment

A 5×5 grid world.
Each cell is either:
0 = Clean
1 = Dirty

Exactly 5 dirty tiles are randomly placed.
Agent begins at the origin (0,0).

## Actuators
Up
Down
Left
Right
Suck

## Sensors
Current tile status (Dirty or Clean).
Current position (x, y).
For Agent B, memory of previously visited locations.

In [1]:
import random

GRID_SIZE = 5
DIRT_COUNT = 5

In [2]:
# -------------------------------
# Environment Creation
# -------------------------------
def create_environment():
    grid = [[0 for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]

    dirty_cells = random.sample(
        [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE)],
        DIRT_COUNT
    )

    for r, c in dirty_cells:
        grid[r][c] = 1

    return grid


def print_grid(grid):
    for row in grid:
        print(row)
    print()


In [3]:
# -------------------------------
# Helper Functions
# -------------------------------
def get_valid_neighbors(position):
    r, c = position
    neighbors = []

    if r > 0:
        neighbors.append((r - 1, c))  # Up

    if r < GRID_SIZE - 1:
        neighbors.append((r + 1, c))  # Down

    if c > 0:
        neighbors.append((r, c - 1))  # Left

    if c < GRID_SIZE - 1:
        neighbors.append((r, c + 1))  # Right

    return neighbors


In [4]:
# -------------------------------
# Agent A: Simple Reflex Agent
# -------------------------------
def simple_reflex_agent(grid, steps=50):
    position = (0, 0)
    score = 0

    print("=== Agent A: Simple Reflex Agent ===")

    for step in range(steps):
        r, c = position

        # Dirty tile -> Suck
        if grid[r][c] == 1:
            grid[r][c] = 0
            score += 10
            action = "SUCK"

        # Clean tile -> Random Move
        else:
            position = random.choice(get_valid_neighbors(position))
            score -= 1
            action = f"MOVE to {position}"

        print(
            f"Step {step+1}: Position={position}, Action={action}, Score={score}"
        )

    return score



In [5]:
# -------------------------------
# Agent B: Model-Based Reflex Agent
# -------------------------------
def model_based_agent(grid, steps=50):
    position = (0, 0)
    visited = {position}
    score = 0

    print("\n=== Agent B: Model-Based Reflex Agent ===")

    for step in range(steps):
        r, c = position

        # Dirty tile -> Suck
        if grid[r][c] == 1:
            grid[r][c] = 0
            score += 10
            action = "SUCK"

        else:
            neighbors = get_valid_neighbors(position)

            unvisited = [
                cell for cell in neighbors
                if cell not in visited
            ]

            if unvisited:
                position = random.choice(unvisited)
            else:
                position = random.choice(neighbors)

            visited.add(position)
            score -= 1
            action = f"MOVE to {position}"

        print(
            f"Step {step+1}: Position={position}, Action={action}, Score={score}"
        )

    return score


In [6]:
# -------------------------------
# Main Program
# -------------------------------
environment = create_environment()

print("Initial Environment")
print_grid(environment)

# Copy environment so both agents
# start with the same room state.
import copy

env_a = copy.deepcopy(environment)
env_b = copy.deepcopy(environment)

score_a = simple_reflex_agent(env_a)
score_b = model_based_agent(env_b)

print("\nFinal Scores")
print("Simple Reflex Agent:", score_a)
print("Model-Based Agent :", score_b)

Initial Environment
[1, 1, 0, 0, 0]
[0, 0, 0, 1, 0]
[0, 0, 0, 0, 0]
[0, 1, 0, 0, 1]
[0, 0, 0, 0, 0]

=== Agent A: Simple Reflex Agent ===
Step 1: Position=(0, 0), Action=SUCK, Score=10
Step 2: Position=(0, 1), Action=MOVE to (0, 1), Score=9
Step 3: Position=(0, 1), Action=SUCK, Score=19
Step 4: Position=(0, 0), Action=MOVE to (0, 0), Score=18
Step 5: Position=(0, 1), Action=MOVE to (0, 1), Score=17
Step 6: Position=(1, 1), Action=MOVE to (1, 1), Score=16
Step 7: Position=(2, 1), Action=MOVE to (2, 1), Score=15
Step 8: Position=(1, 1), Action=MOVE to (1, 1), Score=14
Step 9: Position=(0, 1), Action=MOVE to (0, 1), Score=13
Step 10: Position=(0, 2), Action=MOVE to (0, 2), Score=12
Step 11: Position=(1, 2), Action=MOVE to (1, 2), Score=11
Step 12: Position=(1, 3), Action=MOVE to (1, 3), Score=10
Step 13: Position=(1, 3), Action=SUCK, Score=20
Step 14: Position=(1, 4), Action=MOVE to (1, 4), Score=19
Step 15: Position=(2, 4), Action=MOVE to (2, 4), Score=18
Step 16: Position=(1, 4), Action

# Final Outcome:

In the sample run output provided, Agent A ends with a negative score ($-6$) due to excessive random wandering and wasted movement steps. 

In contrast, Agent B finishes with a positive score ($5$), demonstrating how even simple state memory drastically improves autonomous agent efficiency. 